# RSSI Analyzer — Model 1: Vehicle-tier 1D-CNN (Federated) · **v2**
**v2 = LLM-integration alignment.** The detector + FL loop are unchanged from v1;
the *dataset* and the *export interface* are rebuilt so this model is the join
twin of `vehicle_trust_score_model_v2` / `temporal_gru_seq_v2.1` in the Eq 3.18 fusion.

**Dataset (matches trust_v2):** the attacker-percentage sweep
`pct20_s1 + pct40_s1 + pct100_s1` (mode-9 `sequential_all6`, 259 veh, 4 controllers,
48 s/phase). pct<100 gives a genuine legitimate class; pct100 is the all-attack stress
case. The 6 attack types run **sequentially by phase** (1..6) and, inside each phase,
the active-attacker fraction ramps **20→40→60→80→100** over 5 sub-windows. Only a shared
portion of claimed identities (`identity_manifest.KEEP_FRAC`) is used per run, sampled
leakage-safe by identity so every phase/intensity is still covered.

**Architecture:** (B, 2, W) → Conv1D×2 → GlobalPool → **φ_RSSI ∈ ℝ³²** (pooled, pre-head)
→ Linear(32→1)+σ. The sigmoid head is a training scaffold; the deployed output is the
32-d φ vector (mirrors the GRU's `phi_temp`).

**Data contract (identical to trust_v2's `vehicle_trust_scores_v2.csv`):** one row per
window, keyed **`(claimed_node_id, window_start_seconds, split)`**, carrying
`attack_type∈{0..6}`, `is_sybil∈{0,1}`, `attack_percentage`, `run_id`, and
`active_attack_pct` (EVAL-ONLY — never a model input).

**Output:** `notebooks/outputs/rssi_vehicle_tier.parquet` (grid-snapped, one row per
`(run_id, claimed_node_id, window_start_seconds, split)`) = keys + `phi_rssi_0..31` + `y_hat_i`,
the join twin of `temporal_vehicle_tier.parquet` and `vehicle_trust_scores_v2`.


In [1]:
import os, copy, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import matthews_corrcoef, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit, ShuffleSplit
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import matplotlib.pyplot as plt

BASE_DIR   = '/home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack'
OUTPUT_DIR = f'{BASE_DIR}/ml/Outputs_RSSI'          # model checkpoint / plots / results
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Shared fusion spine: identical identity subsample, leakage-safe split, and 2 s
#    export grid so RSSI / GRU / trust phi tables join on
#    (run_id, claimed_node_id, window_start_seconds, split). Single source of truth.
import sys; sys.path.insert(0, f'{BASE_DIR}/ml/fusion')
import identity_manifest as IDM
MAN          = IDM.load_manifest()                  # notebooks/outputs/split_map.parquet
DATASETS     = IDM.RUN_DIRS                          # the three pct-sweep runs
ARTIFACTS_DIR= str(IDM.ARTIFACTS_DIR)               # fusion contract exports land here
EXPORT_GRID_S= IDM.EXPORT_GRID_S                     # shared 2 s grid
MAX_WINDOWS  = 18_000_000                            # HPC memory guard (see preflight)
# identity subsample fraction lives in identity_manifest.KEEP_FRAC (shared)

# Intensity ladder (mode-9 stepped schedule). active_attack_pct is EVAL-ONLY metadata,
# derived deterministically from beacon time (verified 100% vs the comm-log column in
# trust_v2) — a real vehicle cannot observe attack density, so it is never a feature.
INTENSITY_LADDER = [20, 40, 60, 80, 100]
N_SUBWINDOWS     = 5

FL = {
    'filters'            : 32,   # v2: fixed 32 so phi_RSSI in R^32 aligns with phi_temp
    'kernel_size'        : 5,
    'pooling'            : 'avg',
    'dropout'            : 0.3,
    'lr'                 : 0.005,   # Table 4.5 candidate {0.001, 0.005, 0.01, 0.05};
                                     # baseline for the sensitivity sweep below, not a manual pick
    'batch_size'         : 16,
    'window_W'           : 10,
    'local_epochs'       : 2,
    'mu'                 : 0.01,
    'clip_norm'          : 1.0,     # DP-SGD update clipping bound (L2 norm of client delta)
    'sigma_dp'           : 0.05,    # Table 4.5 candidate {0.05, 0.10, 0.20}; validated by the
                                     # sweep below rather than assumed
    'trim_frac'          : 0.1,
    'fl_rounds'          : 50,
    'patience'           : 5,       # Section 4.4.3: stop when val MCC fails to improve by
                                     # more than min_delta over 5 consecutive rounds
    'min_delta'          : 1e-3,
    'client_sample_frac' : 0.3,     # Table 4.5 candidate {0.1, 0.3, 0.5}; baseline for the sweep
}

# Sensitivity-analysis candidate grids (Section 4.4.3 / Table 4.5) — evaluated
# one-factor-at-a-time, holding the rest of FL fixed at the baseline above.
SENSITIVITY_GRID = {
    'lr'                 : [0.001, 0.005, 0.01, 0.05],
    'sigma_dp'           : [0.05, 0.10, 0.20],
    'client_sample_frac' : [0.3, 0.5],
}

# RSU mid-tier hierarchy (Eq 3.30 zone pre-aggregation, Eq 3.31 global aggregation).
# rssi_verification_log_Dataset.csv carries no per-vehicle serving_rsu_id / zone field
# (checked: no other exported log in outputs/ carries one either), and the ns-3 sim
# itself only instantiates 2-3 physical RSUs (see sybil-attack/configs/*.cfg) — far
# fewer than the paper's 4-zone SDN controller architecture. Vehicles are therefore
# assigned to N_RSUS=4 zones deterministically by vehicle ID modulo N_RSUS, as a
# documented synthetic stand-in for true position-based RSU assignment that matches
# the paper's zone count rather than the physical simulation's RSU count.
N_RSUS = 4

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')
print(f'FL config: {FL}')
print(f'N_RSUS: {N_RSUS} (deterministic vehicle_id % {N_RSUS} zoning — see comment above)')


Device: cuda
FL config: {'filters': 32, 'kernel_size': 5, 'pooling': 'avg', 'dropout': 0.3, 'lr': 0.005, 'batch_size': 16, 'window_W': 10, 'local_epochs': 2, 'mu': 0.01, 'clip_norm': 1.0, 'sigma_dp': 0.05, 'trim_frac': 0.1, 'fl_rounds': 50, 'patience': 5, 'min_delta': 0.001, 'client_sample_frac': 0.3}
N_RSUS: 4 (deterministic vehicle_id % 4 zoning — see comment above)


## Step 1 — Load the pct-sweep, label (7-class), tag run / intensity

In [2]:
def assign_attack_type(times, phase_df):
    '''Phase attack_type (1..6) for each beacon time; 0 before the first phase.
    Matches trust_v2.assign_attack_type (searchsorted on phase start_time).'''
    starts = phase_df['start_time'].to_numpy(dtype=float)
    types  = phase_df['attack_type'].to_numpy()
    idx = np.searchsorted(starts, np.asarray(times, dtype=float), side='right') - 1
    out = np.zeros(len(idx), dtype=np.int64); ok = idx >= 0
    out[ok] = types[idx[ok]]
    return out

def compute_active_attack_pct(times, phase_duration, ladder=INTENSITY_LADDER, n_sub=N_SUBWINDOWS):
    '''EVAL-ONLY: deterministic active-attacker % at time t (trust_v2-identical).'''
    t = np.asarray(times, dtype=float); sw = phase_duration / n_sub
    phase_t = t - (t // phase_duration) * phase_duration
    k = np.clip((phase_t // sw).astype(int), 0, n_sub - 1)
    return np.asarray(ladder)[k]

RSSI_USE = ['time', 'observer_vehicle_id', 'observed_claimed_id', 'observed_real_id', 'rssi_dbm']

def load_run(run_dir):
    meta  = json.load(open(f'{run_dir}/run_meta.json'))
    pct   = int(meta['sybil_attack_percentage'])
    pdur  = float(meta.get('sim_time_per_phase', 48))
    run_id = str(meta.get('run_id', os.path.basename(run_dir)))
    ph = next(p for p in os.listdir(run_dir) if p.startswith('attack_phases'))
    phases = pd.read_csv(f'{run_dir}/{ph}').sort_values('start_time')

    # Memory-safe: stream the ~3 GB rssi log in chunks, keeping ONLY the shared
    # identity subsample (same ids as GRU & trust). Full file never held in RAM.
    df = IDM.read_run_log_filtered(f'{run_dir}/rssi_verification_log.csv', RSSI_USE,
                                   'observed_claimed_id', run_id, MAN)
    df = df.drop_duplicates(subset=['time', 'observer_vehicle_id',
                                    'observed_claimed_id', 'observed_real_id', 'rssi_dbm'])
    for c in ['observer_vehicle_id', 'observed_claimed_id', 'observed_real_id']:
        df[c] = df[c].astype('int32')
    df['is_sybil']    = (df['observed_claimed_id'] != df['observed_real_id']).astype('int8')
    ptype             = assign_attack_type(df['time'].values, phases)
    df['attack_type'] = np.where(df['is_sybil'].values == 1, ptype, 0).astype('int8')

    df['run_id']            = run_id
    df['attack_percentage'] = np.int16(pct)
    df['phase_dur']         = np.float32(pdur)
    df['active_attack_pct'] = compute_active_attack_pct(df['time'].values, pdur).astype('int16')
    print(f'  {run_id:>10} (pct{pct:>3}): {len(df):,} rows  '
          f'kept-ids {df.observed_claimed_id.nunique()}  sybil {int(df.is_sybil.sum()):,}')
    return df

print('Loading pct-sweep (one run at a time)...')
df_raw = pd.concat([load_run(d) for d in DATASETS], ignore_index=True)
print(f'\nTotal rows: {len(df_raw):,}')
print('attack_type (7-class):', df_raw['attack_type'].value_counts().sort_index().to_dict())
print('by attack_percentage :', df_raw['attack_percentage'].value_counts().sort_index().to_dict())
print('by active_attack_pct :', df_raw['active_attack_pct'].value_counts().sort_index().to_dict())


Loading pct-sweep (one run at a time)...


    [safe-read] rssi_verification_log.csv: scanned 53,839,340 rows, kept 27,391,335 (50.9%)


   pct100_s1 (pct100): 11,072,514 rows  kept-ids 824  sybil 2,346,316


    [safe-read] rssi_verification_log.csv: scanned 62,677,278 rows, kept 26,044,070 (41.6%)


    pct20_s1 (pct 20): 11,211,265 rows  kept-ids 277  sybil 1,484,548


    [safe-read] rssi_verification_log.csv: scanned 60,795,090 rows, kept 22,962,976 (37.8%)


    pct40_s1 (pct 40): 9,415,328 rows  kept-ids 428  sybil 1,573,286



Total rows: 31,699,107
attack_type (7-class): {0: 26294957, 1: 1666183, 2: 2994359, 3: 599547, 4: 120015, 5: 24046}
by attack_percentage : {20: 11211265, 40: 9415328, 100: 11072514}
by active_attack_pct : {20: 5720117, 40: 6307958, 60: 6718075, 80: 6557885, 100: 6395072}


## Step 2 — Mean-centre RSSI per (run, observer), clip outliers

In [3]:
# Centre per (run_id, observer): removes the static path-loss offset within each sim.
df = df_raw.sort_values(['run_id', 'observer_vehicle_id', 'observed_claimed_id', 'time']).copy()
df['rssi_centred'] = (df.groupby(['run_id', 'observer_vehicle_id'])['rssi_dbm']
                      .transform(lambda s: s - s.mean()).astype('float32'))
p1, p99 = np.percentile(df['rssi_centred'].values, [1, 99])
df['rssi_centred'] = np.clip(df['rssi_centred'].values, p1, p99).astype('float32')
df = df.reset_index(drop=True)
print(f'RSSI centred clip: [{p1:.2f}, {p99:.2f}] dBm   rows: {len(df):,}')
df[['rssi_dbm', 'rssi_centred']].describe().round(3)


RSSI centred clip: [-110.35, 13.53] dBm   rows: 31,699,107


,rssi_dbm,rssi_centred
count,3.169911e+07,3.169911e+07
mean,2.836800e+01,-1.700000e-01
std,3.410000e+01,1.641700e+01
min,-8.200000e+01,-1.103500e+02
25%,4.000000e+01,1.992000e+00
50%,4.000000e+01,2.481000e+00
75%,4.000000e+01,3.114000e+00
max,4.000000e+01,1.353200e+01


## Step 3 — Sliding W=10 windows per (run, observer, claimed) · seconds-keyed

In [4]:
class RSSIWindowDataset(Dataset):
    '''Sliding (2,W) windows per (run_id, observer, claimed_id): ch0 mean-centred RSSI,
    ch1 first difference. Each window records the fusion join keys:
      claimed_node_id, receiver_id, window_start_seconds (data contract),
      run_id, attack_percentage, attack_type(7-class @ window start), active_attack_pct.
    Split grouping key = (run_id, observed_real_id) — matches trust_v2 (no real-vehicle
    leak across splits).'''
    def __init__(self, df, W, keep_prob=1.0):
        _wr = np.random.default_rng(SEED)
        wins, y, claimed, obs, real = [], [], [], [], []
        run, sidx, ssec, atype, pct, aap, gkey = [], [], [], [], [], [], []
        for (rid, o, c), grp in df.groupby(['run_id', 'observer_vehicle_id', 'observed_claimed_id'], sort=False):
            grp = grp.sort_values('time')
            s   = grp['rssi_centred'].values.astype(np.float32)
            t   = grp['time'].values.astype(np.float64)
            at  = grp['attack_type'].values; ap = grp['active_attack_pct'].values
            lab = int(grp['is_sybil'].iloc[0]); rid_real = int(grp['observed_real_id'].iloc[0])
            p   = int(grp['attack_percentage'].iloc[0])
            n = len(s)
            if n < W:
                s = np.pad(s, (W - n, 0), mode='edge'); n = W
            d = np.diff(s, prepend=s[0]).astype(np.float32)
            for st in range(n - W + 1):
                if keep_prob < 1.0 and _wr.random() >= keep_prob:
                    continue   # memory-budgeted window subsample (seeded)
                j = min(st, len(t) - 1)
                wins.append(np.stack([s[st:st + W], d[st:st + W]], axis=0))
                y.append(lab); claimed.append(int(c)); obs.append(int(o)); real.append(rid_real)
                run.append(rid); sidx.append(st); ssec.append(float(t[j]))
                atype.append(int(at[j])); pct.append(p); aap.append(int(ap[j]))
                gkey.append(f'{rid}__vid{rid_real}')
        self.X            = torch.tensor(np.array(wins), dtype=torch.float32)
        self.y            = torch.tensor(y, dtype=torch.float32)
        self.claimed_ids  = np.array(claimed, dtype=np.int64)
        self.obs_ids      = np.array(obs, dtype=np.int64)          # receiver_id
        self.real_ids     = np.array(real, dtype=np.int64)
        self.run_ids      = np.array(run, dtype=object)
        self.start_idx    = np.array(sidx, dtype=np.int64)
        self.start_sec    = np.array(ssec, dtype=np.float64)       # window_start_seconds
        self.attack_types = np.array(atype, dtype=np.int64)
        self.attack_pct   = np.array(pct, dtype=np.int64)
        self.active_pct   = np.array(aap, dtype=np.int64)          # EVAL-ONLY
        self.groups       = np.array(gkey, dtype=object)           # split group key
        # FL client = one OBU within one run (trust_v2 groups clients by (run, observer))
        self.client_ids   = np.array([f'{r}:{o}' for r, o in zip(run, obs)], dtype=object)

    def __len__(self):          return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

print(f'Building windows W={FL["window_W"]} ...')
_kp = min(1.0, MAX_WINDOWS / max(len(df), 1))
dataset = RSSIWindowDataset(df, FL['window_W'], keep_prob=_kp)
print(f'window keep_prob={_kp:.3f} (cap {MAX_WINDOWS:,})')
n_s = int(dataset.y.sum())
print(f'Total windows: {len(dataset):,}   Sybil: {n_s:,}   Legit: {len(dataset)-n_s:,}')
print(f'Input shape: {tuple(dataset.X[0].shape)}   split groups: {len(np.unique(dataset.groups)):,}'
      f'   FL clients: {len(np.unique(dataset.client_ids)):,}')


Building windows W=10 ...


window keep_prob=0.568 (cap 18,000,000)
Total windows: 17,837,172   Sybil: 2,974,973   Legit: 14,862,199


Input shape: (2, 10)   split groups: 716   FL clients: 926


## Step 4 — Group-stratified 70/15/15 split (by real vehicle, within (pct, type))

In [5]:
# Split comes from the SHARED manifest (grouped by real vehicle; identical across
# RSSI / GRU / trust). We no longer compute our own split.
split_arr = IDM.split_of(dataset.run_ids, dataset.claimed_ids, MAN)
dataset.split_arr = split_arr
idx = np.arange(len(dataset))
train_ds = Subset(dataset, idx[split_arr == 'train'])
val_ds   = Subset(dataset, idx[split_arr == 'val'])
test_ds  = Subset(dataset, idx[split_arr == 'test'])
print(f'Train: {len(train_ds):,}   Val: {len(val_ds):,}   Test: {len(test_ds):,}')
print('split counts:', {k: int((split_arr == k).sum()) for k in ['train', 'val', 'test']})

# leakage guard via the shared vehicle map
_vk = MAN.set_index(['run_id', 'claimed_node_id'])['vehicle_key']
veh = np.array([_vk.get((r, int(c)), f'{r}?{c}') for r, c in zip(dataset.run_ids, dataset.claimed_ids)], dtype=object)
assert not (set(veh[split_arr=='train']) & set(veh[split_arr=='test'])), 'identity leak!'

train_labels = dataset.y.numpy()[train_ds.indices]
n_neg = (train_labels == 0).sum(); n_pos = (train_labels == 1).sum()
pos_weight_val = n_neg / max(1, n_pos)
pos_w = torch.tensor([pos_weight_val], dtype=torch.float32).to(DEVICE)
print(f'pos_weight: {pos_weight_val:.2f}   (train legit {int(n_neg):,} / sybil {int(n_pos):,})')

val_loader  = DataLoader(val_ds,  batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)


Train: 13,216,824   Val: 2,101,909   Test: 2,518,439
split counts: {'train': 13216824, 'val': 2101909, 'test': 2518439}


pos_weight: 5.61   (train legit 11,216,278 / sybil 2,000,546)


## Step 5 — Partition Training Set into FL Clients (one per OBU)

In [6]:
train_cli  = dataset.client_ids[train_ds.indices]
unique_cli = np.unique(train_cli)

client_loaders = {}
for cli in unique_cli:
    idx_local = np.where(train_cli == cli)[0]
    if len(idx_local) < FL['batch_size']:
        continue
    sub = Subset(train_ds, idx_local)
    client_loaders[cli] = DataLoader(sub, batch_size=FL['batch_size'], shuffle=True, drop_last=True)

print(f'OBU-runs total: {len(unique_cli)}   FL clients (sufficient data): {len(client_loaders)}')
szs = [len(l.dataset) for l in client_loaders.values()]
print(f'Windows per client — min:{min(szs)}  median:{int(np.median(szs))}  max:{max(szs)}')


OBU-runs total: 916   FL clients (sufficient data): 900
Windows per client — min:17  median:1568  max:78124


In [7]:
# Each FL client is 'run_id:observer'; zone by the observer id modulo N_RSUS
# (synthetic zoning stand-in, see N_RSUS comment). run_id keeps clients from
# different pct runs distinct while still mapping to the paper's 4 controller zones.
def _obs_of(cli):  # 'pct40_s1:137' -> 137
    return int(str(cli).split(':')[-1])
client_zone = {cli: _obs_of(cli) % N_RSUS for cli in unique_cli}
zone_counts = pd.Series(list(client_zone.values())).value_counts().sort_index()
print(f'Zone assignment (observer_id % {N_RSUS}):')
for z, cnt in zone_counts.items():
    print(f'  RSU zone {z}: {cnt} OBU-runs')


Zone assignment (observer_id % 4):
  RSU zone 0: 240 OBU-runs
  RSU zone 1: 231 OBU-runs
  RSU zone 2: 218 OBU-runs
  RSU zone 3: 227 OBU-runs


## Step 6 — 1D-CNN Model

In [8]:
class RSSIAnalyzerCNN(nn.Module):
    '''Compact federated client CNN (Section 4.4.2 / Eq 3.6). Pooled hidden h in R^F
    is phi_RSSI(v_i) for Eq 3.18; F=32 matches phi_temp. Linear(F,1)+sigmoid is a
    training scaffold — the deployed export is phi (the 32-d vector).'''
    def __init__(self, F, K, pooling, dropout, in_channels=2):
        super().__init__(); P = K // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, F, K, padding=P), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(F, F, K, padding=P), nn.ReLU(), nn.Dropout(dropout))
        self.pool = pooling
        self.head = nn.Linear(F, 1)

    def forward(self, x, return_phi=False):
        h   = self.conv(x)
        phi = h.mean(-1) if self.pool == 'avg' else h.max(-1).values   # (B, F) = phi_RSSI
        logit = self.head(phi).squeeze(-1)
        return (logit, phi) if return_phi else logit

    def phi(self, x):
        return torch.sigmoid(self.forward(x))   # scalar score for MCC/threshold helpers

model = RSSIAnalyzerCNN(FL['filters'], FL['kernel_size'], FL['pooling'], FL['dropout'],
                        in_channels=dataset.X.shape[1]).to(DEVICE)
assert FL['filters'] == 32, 'phi_RSSI must be 32-d to align with phi_temp for Eq 3.18'
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model); print(f'Trainable parameters: {n_p:,}   phi_RSSI dim: {FL["filters"]}')


RSSIAnalyzerCNN(
  (conv): Sequential(
    (0): Conv1d(2, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
  )
  (head): Linear(in_features=32, out_features=1, bias=True)
)
Trainable parameters: 5,537   phi_RSSI dim: 32


## Step 7 — FL Utilities: FedProx, DP Noise, Aggregation

In [9]:
def fedprox_local_train(loader, global_sd, model, config, device, pos_w):
    '''Local steps with FedProx proximal term (Eq 3.40).'''
    model.load_state_dict(copy.deepcopy(global_sd))
    opt   = optim.Adam(model.parameters(), lr=config['lr'])
    crit  = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    g_params = [global_sd[k].to(device) for k in global_sd]
    model.train()
    for _ in range(config['local_epochs']):
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(X), y)
            prox = sum((p - g).norm() ** 2
                       for p, g in zip(model.parameters(), g_params))
            (loss + (config['mu'] / 2) * prox).backward()
            opt.step()
    return {k: v.detach().cpu() for k, v in model.state_dict().items()}


def clip_and_noise_delta(local_sd, global_sd, clip_norm, sigma):
    '''
    DP-SGD-style update sanitization (Eq 3.40), fixed:
      1) Clip the *whole client delta* to L2 norm <= clip_norm — bounds each
         client's contribution regardless of local data/gradient scale.
      2) Add Gaussian noise calibrated to the clip bound (sigma * clip_norm),
         not a flat sigma applied straight to raw weights. Flat noise on a
         1.4k-parameter model swamped the signal and made training unstable
         (validation MCC bounced round to round instead of converging).
    '''
    delta = {k: local_sd[k] - global_sd[k] for k in global_sd}
    total_norm = torch.sqrt(sum((v.float() ** 2).sum() for v in delta.values()))
    scale = min(1.0, clip_norm / (total_norm.item() + 1e-8))
    noisy_delta = {
        k: v * scale + torch.randn_like(v) * (sigma * clip_norm)
        for k, v in delta.items()
    }
    return noisy_delta


def trimmed_mean_agg(updates, sizes, trim_frac):
    '''
    Two-step aggregation (Eq 3.40 + Eq 3.41):
      Step 1 — Byzantine-robust trimmed mean (Eq 3.41):
               rank clients by L2 norm of their delta,
               drop the bottom and top trim_frac fraction.
      Step 2 — Size-weighted average over the surviving clients (Eq 3.40):
               larger OBU datasets contribute proportionally more.
    '''
    norms  = [sum(v.norm().item() for v in u.values()) for u in updates]
    n      = len(norms)
    trim_n = max(0, int(n * trim_frac))
    idx    = sorted(range(n), key=lambda i: norms[i])
    sel    = idx[trim_n : n - trim_n] if n - 2 * trim_n >= 1 else idx

    sel_sizes = [sizes[i] for i in sel]
    total     = sum(sel_sizes)
    agg = {}
    for k in updates[0]:
        agg[k] = sum(
            (sel_sizes[j] / total) * updates[sel[j]][k]
            for j in range(len(sel))
        )
    return agg


def hierarchical_aggregate(client_deltas, client_sizes, client_ids, client_zone, n_rsus, trim_frac):
    '''
    Two-level federated aggregation matching the report's vehicle -> RSU -> SDN
    controller hierarchy (Eq 3.30 zone/RSU-tier pre-aggregation, Eq 3.31 global
    aggregation at the controller). Replaces flat trimmed_mean_agg over all
    sampled clients at once, which skipped the RSU tier entirely and forced the
    controller to reconcile all clients' non-IID gradients simultaneously.

    Step 1 (RSU tier)  — within each zone, Byzantine-robust trimmed-mean +
                          size-weighted average of that zone's sampled clients'
                          deltas -> one pre-aggregated zone delta per RSU.
    Step 2 (Controller) — trimmed-mean + size-weighted average of the
                          surviving zone deltas -> global delta.
    '''
    zone_deltas, zone_sizes = {}, {}
    for z in range(n_rsus):
        idx = [i for i, cid in enumerate(client_ids) if client_zone[cid] == z]
        if not idx:
            continue
        zone_deltas[z] = trimmed_mean_agg(
            [client_deltas[i] for i in idx], [client_sizes[i] for i in idx], trim_frac)
        zone_sizes[z] = sum(client_sizes[i] for i in idx)

    zones = list(zone_deltas.keys())
    if len(zones) == 1:
        return zone_deltas[zones[0]]
    return trimmed_mean_agg(
        [zone_deltas[z] for z in zones], [zone_sizes[z] for z in zones], trim_frac)


def compute_mcc(m, loader, thresh=0.5):
    m.eval()
    phis, ys = [], []
    with torch.no_grad():
        for X, y in loader:
            phis.extend(m.phi(X.to(DEVICE)).cpu().numpy())
            ys.extend(y.numpy().astype(int))
    pred = (np.array(phis) >= thresh).astype(int)
    return matthews_corrcoef(ys, pred) if len(set(pred)) > 1 else 0.0

print('FL utilities ready.')

FL utilities ready.


In [10]:
def run_federated_training(config, client_loaders, client_zone, n_rsus,
                            val_loader, pos_w, in_channels, seed=SEED, verbose=True):
    '''
    Runs one full federated training session for a given hyperparameter config.
    Returns (best_state_dict, best_val_mcc, history) — used both for the main
    run and for the one-factor-at-a-time sensitivity sweep, so every sweep
    candidate goes through the identical two-level (RSU -> controller)
    aggregation and early-stopping rule.
    '''
    torch.manual_seed(seed); np.random.seed(seed)
    m = RSSIAnalyzerCNN(config['filters'], config['kernel_size'],
                        config['pooling'], config['dropout'],
                        in_channels=in_channels).to(DEVICE)

    global_sd    = {k: v.cpu() for k, v in m.state_dict().items()}
    best_sd      = copy.deepcopy(global_sd)
    best_val_mcc = -1.0
    no_improve   = 0
    history      = []
    client_ids   = list(client_loaders.keys())
    n_sample     = max(1, int(len(client_ids) * config['client_sample_frac']))

    if verbose:
        print(f'Total clients: {len(client_ids)}   Sampled per round: {n_sample} '
              f'({config["client_sample_frac"]*100:.0f}%)   RSU zones: {n_rsus}')
        print(f'Rounds: {config["fl_rounds"]}   Patience: {config["patience"]}')
        print(f'{"Round":>6}  {"Clients":>8}  {"Val MCC":>9}  {"Best MCC":>9}')
        print('-' * 42)

    for rnd in range(1, config['fl_rounds'] + 1):
        rnd_clients   = list(np.random.choice(client_ids, size=n_sample, replace=False))
        client_deltas = []
        client_sizes  = []

        for obs_id in rnd_clients:
            local_sd = fedprox_local_train(
                client_loaders[obs_id], global_sd, m, config, DEVICE, pos_w)
            delta = clip_and_noise_delta(
                local_sd, global_sd, config['clip_norm'], config['sigma_dp'])
            client_deltas.append(delta)
            client_sizes.append(len(client_loaders[obs_id].dataset))

        # Two-level hierarchy: RSU-tier zone pre-aggregation (Eq 3.30), then
        # controller-tier global aggregation over the zone deltas (Eq 3.31).
        agg_delta = hierarchical_aggregate(
            client_deltas, client_sizes, rnd_clients, client_zone, n_rsus, config['trim_frac'])

        global_sd = {k: global_sd[k] + agg_delta[k] for k in global_sd}
        m.load_state_dict({k: v.to(DEVICE) for k, v in global_sd.items()})

        val_mcc = compute_mcc(m, val_loader)
        history.append({'round': rnd, 'val_mcc': float(val_mcc),
                        'clients_used': len(rnd_clients)})
        if verbose and (rnd % 5 == 0 or rnd == 1):
            print(f'{rnd:>6}  {len(rnd_clients):>8}  {val_mcc:>9.4f}  {best_val_mcc:>9.4f}')

        if val_mcc - best_val_mcc > config['min_delta']:
            best_val_mcc = val_mcc
            best_sd      = copy.deepcopy(global_sd)
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= config['patience']:
                if verbose:
                    print(f'\nEarly stop at round {rnd}  (best val MCC={best_val_mcc:.4f})')
                break

    return best_sd, best_val_mcc, history

print('run_federated_training defined.')

run_federated_training defined.


## Step 7b — Hyperparameter Sensitivity Sweep (Section 4.4.3 methodology)

One-factor-at-a-time: vary a single hyperparameter across its Table 4.5 candidate
set, holding the rest at baseline, and keep whichever value maximises validation
MCC. Each candidate is trained at a reduced round budget (`sweep_rounds`) to keep
the sweep computationally tractable on CPU; the winning combination is then
re-run at the full `fl_rounds` budget in the main training run below.

In [11]:
def sweep_one_factor(param_name, candidates, base_config, sweep_rounds=15):
    results = []
    for val in candidates:
        cfg = dict(base_config)
        cfg[param_name] = val
        cfg['fl_rounds'] = sweep_rounds
        _, val_mcc, _ = run_federated_training(
            cfg, client_loaders, client_zone, N_RSUS, val_loader, pos_w,
            in_channels=dataset.X.shape[1], verbose=False)
        results.append({'param': param_name, 'value': val, 'val_mcc': val_mcc})
        print(f'  {param_name}={val:<8}  val MCC={val_mcc:.4f}')
    best = max(results, key=lambda r: r['val_mcc'])
    print(f'  -> best {param_name} = {best["value"]}  (val MCC={best["val_mcc"]:.4f})\n')
    return best['value'], results

print('=== Sensitivity sweep (one-factor-at-a-time, Section 4.4.3) ===\n')
sweep_results = []
tuned_FL = dict(FL)

print('Learning rate:')
best_lr, r = sweep_one_factor('lr', SENSITIVITY_GRID['lr'], tuned_FL)
tuned_FL['lr'] = best_lr; sweep_results += r

print('DP noise scale (sigma_dp):')
best_sigma, r = sweep_one_factor('sigma_dp', SENSITIVITY_GRID['sigma_dp'], tuned_FL)
tuned_FL['sigma_dp'] = best_sigma; sweep_results += r

print('Client sample fraction:')
best_frac, r = sweep_one_factor('client_sample_frac', SENSITIVITY_GRID['client_sample_frac'], tuned_FL)
tuned_FL['client_sample_frac'] = best_frac; sweep_results += r

FL = tuned_FL
print(f'Winning config after sweep: lr={FL["lr"]}, sigma_dp={FL["sigma_dp"]}, '
      f'client_sample_frac={FL["client_sample_frac"]}')

pd.DataFrame(sweep_results).to_csv(f'{OUTPUT_DIR}/fl_sensitivity_sweep.csv', index=False)
print(f'Saved -> {OUTPUT_DIR}/fl_sensitivity_sweep.csv')

=== Sensitivity sweep (one-factor-at-a-time, Section 4.4.3) ===

Learning rate:


  lr=0.001     val MCC=0.0027


  lr=0.005     val MCC=0.0000


  lr=0.01      val MCC=0.0000


  lr=0.05      val MCC=0.0000
  -> best lr = 0.001  (val MCC=0.0027)

DP noise scale (sigma_dp):


  sigma_dp=0.05      val MCC=0.0027


  sigma_dp=0.1       val MCC=0.0000


  sigma_dp=0.2       val MCC=0.0465
  -> best sigma_dp = 0.2  (val MCC=0.0465)

Client sample fraction:


  client_sample_frac=0.3       val MCC=0.0465


  client_sample_frac=0.5       val MCC=0.0523
  -> best client_sample_frac = 0.5  (val MCC=0.0523)

Winning config after sweep: lr=0.001, sigma_dp=0.2, client_sample_frac=0.5
Saved -> /home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack/ml/Outputs_RSSI/fl_sensitivity_sweep.csv


## Step 8 — Federated Training Loop

In [12]:
best_sd, best_val_mcc, history = run_federated_training(
    FL, client_loaders, client_zone, N_RSUS, val_loader, pos_w,
    in_channels=dataset.X.shape[1])
model.load_state_dict({k: v.to(DEVICE) for k, v in best_sd.items()})
print(f'\nBest val MCC: {best_val_mcc:.4f}')

Total clients: 900   Sampled per round: 450 (50%)   RSU zones: 4
Rounds: 50   Patience: 5
 Round   Clients    Val MCC   Best MCC
------------------------------------------


     1       450     0.0523    -1.0000


     5       450     0.0000     0.0523



Early stop at round 6  (best val MCC=0.0523)

Best val MCC: 0.0523


## Step 9 — Per-Round MCC Convergence Curve (Eq 3.31)

In [13]:
rnds_   = [h['round']   for h in history]
mccs_   = [h['val_mcc'] for h in history]

plt.figure(figsize=(9, 4))
plt.plot(rnds_, mccs_, color='darkorange', lw=2)
plt.axhline(best_val_mcc, color='red', ls='--', alpha=0.6,
            label=f'Best MCC = {best_val_mcc:.4f}')
plt.xlabel('FL Round'); plt.ylabel('Validation MCC')
plt.title('1D-CNN Federated — Per-Round MCC Convergence (Eq 3.31)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fl_mcc_convergence.png', dpi=150)
plt.show()
print(f'Saved: {OUTPUT_DIR}/fl_mcc_convergence.png')

Saved: /home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack/ml/Outputs_RSSI/fl_mcc_convergence.png


## Step 10 — Test Evaluation (threshold tuned on validation)

**Diagnostic only — not the system performance metric.** φ_RSSI is a feature-extractor
output that feeds `ŷ_i` (Eq 3.26), fused with φ_temp (Temporal Analyzer) and φ_trust
(Trust Analyzer). Neither of those two analyzers exists in this repo yet, so `ŷ_i`
cannot be evaluated here. The MCC below is thresholding φ_RSSI directly, purely as an
intermediate sanity check on this analyzer in isolation — it must not be reported or
compared as if it were system-level performance.

In [14]:
def get_probs(m, loader):
    m.eval(); phis, ys = [], []
    with torch.no_grad():
        for X, y in loader:
            phis.extend(m.phi(X.to(DEVICE)).cpu().numpy())
            ys.extend(y.numpy().astype(int))
    return np.array(phis), np.array(ys)

val_phis, val_ys = get_probs(model, val_loader)
best_t, best_mcc_t = 0.5, -1.0
for t in np.arange(0.05, 0.96, 0.05):
    m_ = matthews_corrcoef(val_ys, (val_phis >= t).astype(int))
    if m_ > best_mcc_t:
        best_mcc_t, best_t = m_, float(t)
print(f'Best threshold (val): {best_t:.2f}  val MCC={best_mcc_t:.4f}')

test_phis, test_ys = get_probs(model, test_loader)
y_pred   = (test_phis >= best_t).astype(int)
test_mcc = matthews_corrcoef(test_ys, y_pred)
cm       = confusion_matrix(test_ys, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'\nTest MCC : {test_mcc:.4f}')
print(f'Sybil Recall: {tp/(tp+fn):.4f}   False Alarm: {fp/(fp+tn):.4f}')
print(classification_report(test_ys, y_pred,
      target_names=['Legitimate', 'Sybil'], zero_division=0))

Best threshold (val): 0.10  val MCC=0.0757



Test MCC : 0.0745
Sybil Recall: 0.9981   False Alarm: 0.9664
              precision    recall  f1-score   support

  Legitimate       0.99      0.03      0.06   2059094
       Sybil       0.19      1.00      0.32    459345

    accuracy                           0.21   2518439
   macro avg       0.59      0.52      0.19   2518439
weighted avg       0.84      0.21      0.11   2518439



In [15]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im)
ax.set(xticks=[0,1], yticks=[0,1],
       xticklabels=['Legitimate','Sybil'], yticklabels=['Legitimate','Sybil'],
       xlabel='Predicted', ylabel='Actual',
       title=f'CNN Confusion Matrix  MCC={test_mcc:.4f}')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=13,
                color='white' if cm[i,j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cnn_confusion_matrix.png', dpi=150)
plt.show()

## Step 11 — Export φ_RSSI (32-d) — canonical fusion contract
Twin of trust_v2's `vehicle_trust_scores_v2.csv`: one row per window keyed
**`(claimed_node_id, window_start_seconds, split)`**, with `run_id` /
`attack_percentage` disambiguating the three sims. `fl/build_context_vector.py`
(Eq 3.31) as-of-joins RSSI / temporal / trust on these keys.

In [16]:
@torch.no_grad()
def extract_phi_matrix(m, ds, batch=512):
    m.eval(); loader = DataLoader(ds, batch_size=batch, shuffle=False)
    phis, logits = [], []
    for X, _ in loader:
        logit, phi = m(X.to(DEVICE), return_phi=True)
        logits.append(logit.cpu().numpy()); phis.append(phi.cpu().numpy())
    return np.concatenate(phis, 0), np.concatenate(logits, 0)

phi_mat, logit_vec = extract_phi_matrix(model, dataset)          # (N,32), (N,)
y_hat_i = (1.0 / (1.0 + np.exp(-logit_vec))).astype(np.float32)

# Per-window frame, then snap window_start_seconds to the shared 2 s grid and pool —
# one row per (run_id, claimed_node_id, grid-cell, split), matching GRU & trust so the
# three phi tables inner-join cleanly for Eq 3.18 / Eq 3.31.
w = pd.DataFrame({
    'run_id'            : dataset.run_ids,
    'attack_percentage': dataset.attack_pct.astype(int),
    'claimed_node_id'  : dataset.claimed_ids.astype(int),
    'window_start_seconds': IDM.snap_grid(dataset.start_sec),
    'split'            : dataset.split_arr,
    'y_hat_i'          : y_hat_i,
    'attack_type'      : dataset.attack_types.astype(int),
    'is_sybil'         : dataset.y.numpy().astype(int),
    'active_attack_pct': dataset.active_pct.astype(int),
})
for i in range(phi_mat.shape[1]):
    w[f'phi_rssi_{i}'] = phi_mat[:, i].astype(np.float32)

key = ['run_id', 'attack_percentage', 'claimed_node_id', 'window_start_seconds', 'split']
phi_cols = [f'phi_rssi_{i}' for i in range(phi_mat.shape[1])]
agg = {**{cc: 'mean' for cc in phi_cols}, 'y_hat_i': 'mean',
       'is_sybil': 'max', 'active_attack_pct': 'first'}
pooled = w.groupby(key, as_index=False).agg(agg)
at = (w.groupby(key)['attack_type']
        .agg(lambda s: int(pd.Series(s[s>0]).mode().iloc[0]) if (s>0).any() else 0)
        .reset_index())
out = pooled.merge(at, on=key)
out = out[key + ['is_sybil', 'attack_type', 'active_attack_pct', 'y_hat_i'] + phi_cols]

OUT_PARQUET = f'{ARTIFACTS_DIR}/rssi_vehicle_tier.parquet'
out.to_parquet(OUT_PARQUET, index=False)
print(f'Saved {OUT_PARQUET}  {out.shape}')
print('key cols:', key)

# Per-(run, claimed identity) roll-up (no real_id leakage), mirrors GRU y_hat_i_*.
veh = (out.groupby(['run_id', 'claimed_node_id'])
       .agg(attack_percentage=('attack_percentage', 'first'),
            y_hat_i_mean=('y_hat_i', 'mean'), y_hat_i_max=('y_hat_i', 'max'),
            attack_type=('attack_type', 'max'), is_sybil=('is_sybil', 'max'))
       .reset_index())
veh.to_csv(f'{OUTPUT_DIR}/y_hat_i_rssi.csv', index=False)
print(f'Saved y_hat_i_rssi.csv  {veh.shape}')
out.head(3)


Saved /home/sdvn_defense_sybil/35_sybil_attack_project/ns-allinone-3.35/ns-3.35/sybil-attack/notebooks/outputs/rssi_vehicle_tier.parquet  (51092, 41)
key cols: ['run_id', 'attack_percentage', 'claimed_node_id', 'window_start_seconds', 'split']
Saved y_hat_i_rssi.csv  (1491, 7)


,run_id,attack_percentage,claimed_node_id,window_start_seconds,split,is_sybil,attack_type,active_attack_pct,y_hat_i,phi_rssi_0,...,phi_rssi_22,phi_rssi_23,phi_rssi_24,phi_rssi_25,phi_rssi_26,phi_rssi_27,phi_rssi_28,phi_rssi_29,phi_rssi_30,phi_rssi_31
0,pct100_s1,100,8,48.0,test,0,0,20,0.495612,0.406874,...,0.047917,0.046476,0.105130,0.008902,0.000006,0.140405,0.285694,0.642271,0.232150,0.000004
1,pct100_s1,100,8,50.0,test,0,0,20,0.461067,0.126289,...,0.506171,0.208212,0.072633,0.013376,0.362027,0.079173,0.030058,0.506143,0.063690,0.077414
2,pct100_s1,100,8,52.0,test,0,0,20,0.382348,0.432965,...,1.584940,0.537968,0.109592,0.000000,1.552460,0.380757,0.100158,1.586374,0.077763,0.307129


## Step 12 — Save Model & Results

In [17]:
torch.save({'state_dict': model.state_dict(), 'fl_config': FL,
            'threshold': best_t, 'best_val_mcc': best_val_mcc},
           f'{OUTPUT_DIR}/rssi_cnn_federated_v2.pt')

results = {
    'model'         : 'RSSIAnalyzerCNN_Federated_v2',
    'phi_dim'       : FL['filters'],
    'datasets'      : [os.path.basename(d) for d in DATASETS],
    'subsample_frac': IDM.KEEP_FRAC,
    'test_mcc'      : round(float(test_mcc), 6),
    'val_mcc'       : round(float(best_val_mcc), 6),
    'threshold'     : round(float(best_t), 4),
    'sybil_recall'  : round(float(tp / (tp + fn)), 6),
    'false_alarm'   : round(float(fp / (fp + tn)), 6),
    'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    'fl_rounds_run' : len(history),
    'fl_config'     : FL,
}
with open(f'{OUTPUT_DIR}/cnn_results_v2.json', 'w') as f:
    json.dump(results, f, indent=2)
with open(f'{OUTPUT_DIR}/fl_history_v2.json', 'w') as f:
    json.dump(history, f, indent=2)

print('=== OUTPUTS ===')
for fn_ in ['rssi_cnn_federated.pt', 'cnn_results_v2.json', 'fl_history_v2.json',
            'fl_mcc_convergence.png', 'cnn_confusion_matrix.png',
            'phi_rssi_federated.csv', 'y_hat_i_rssi.csv']:
    p = f'{OUTPUT_DIR}/{fn_}'
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}]  {fn_}')
print(f'\nTest MCC: {test_mcc:.4f}   Val MCC: {best_val_mcc:.4f}')


=== OUTPUTS ===
  [OK]  rssi_cnn_federated.pt
  [OK]  cnn_results_v2.json
  [OK]  fl_history_v2.json
  [OK]  fl_mcc_convergence.png
  [OK]  cnn_confusion_matrix.png
  [MISSING]  phi_rssi_federated.csv
  [OK]  y_hat_i_rssi.csv

Test MCC: 0.0745   Val MCC: 0.0523
